In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Fraud Detection In Bitcoin Transaction Network

In [ ]:
!pip install kagglehub torch torchvision torchaudio
!pip install torch-geometric
!pip install pandas numpy scikit-learn networkx
!pip install ripser persim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 5.7 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18104 sha256=8e4ca726757f4a7d363a072d1c1f3e421111371cd7aed1ab1ec1cfbc1eeb59d3
  Stored in directory: /root/.cache/pip/wheels/2a/fd/fe/f4b8fd82894e1d9e04040ef41dc5ae6eb7a8e9b0ef5a9402fe
Successfully built hopcroftkarp


In [ ]:
# ================================
# 1. IMPORTS
# ================================
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import networkx as nx
from torch_geometric.data import Data
from torch_geometric.nn import GATConv

from ripser import ripser

In [ ]:
# ================================
# 2. LOAD DATA
# ================================

import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("ellipticco/elliptic-data-set")

# Print the contents of the downloaded directory to understand its structure
print(f"Contents of downloaded path ({path}):")
!ls -R {path}

# The actual data is located in a subdirectory named 'elliptic_bitcoin_dataset'
base_path = os.path.join(path, "elliptic_bitcoin_dataset")

# Print base_path to help debugging in case of further errors
print(f"Attempting to load data from: {base_path}")

features = pd.read_csv(f"{base_path}/elliptic_txs_features.csv", header=None)
edges = pd.read_csv(f"{base_path}/elliptic_txs_edgelist.csv")
classes = pd.read_csv(f"{base_path}/elliptic_txs_classes.csv")

print("Data loaded successfully!")

Using Colab cache for faster access to the 'elliptic-data-set' dataset.
Contents of downloaded path (/kaggle/input/elliptic-data-set):
/kaggle/input/elliptic-data-set:
elliptic_bitcoin_dataset

/kaggle/input/elliptic-data-set/elliptic_bitcoin_dataset:
elliptic_txs_classes.csv  elliptic_txs_edgelist.csv  elliptic_txs_features.csv
Attempting to load data from: /kaggle/input/elliptic-data-set/elliptic_bitcoin_dataset
Data loaded successfully!


In [ ]:
# ================================
# 3. PREPROCESS
# ================================
features.columns = ["txId", "time_step"] + [f"f{i}" for i in range(165)]

data = features.merge(classes, on="txId")

# remove unknown
data = data[data["class"] != "unknown"]
data["class"] = data["class"].map({"1": 1, "2": 0})

# features
time_step = data["time_step"]
X = data.drop(["txId", "time_step", "class"], axis=1).values
y = data["class"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print("Preprocessing done!")

Preprocessing done!


In [ ]:
# ================================
# 4. GRAPH CONSTRUCTION
# ================================
valid_nodes = set(data["txId"])

edge_list = []
for _, row in edges.iterrows():
    if row["txId1"] in valid_nodes and row["txId2"] in valid_nodes:
        edge_list.append((row["txId1"], row["txId2"]))

G = nx.DiGraph()
G.add_edges_from(edge_list)

node_map = {tx: i for i, tx in enumerate(data["txId"])}

edge_index = []
for u, v in edge_list:
    edge_index.append([node_map[u], node_map[v]])

edge_index = torch.tensor(edge_index).t().contiguous()

print("Graph built!")

Graph built!


In [ ]:
# ================================
# 5. TDA FUNCTION (ADD HERE)
# ================================

def compute_tda(subgraph):
    nodes = list(subgraph.nodes())

    if len(nodes) < 3:
        return np.zeros(5)

    try:
        A = nx.to_numpy_array(subgraph)
        dist = 1 - A

        dgms = ripser(dist, distance_matrix=True, maxdim=1)['dgms']
        h1 = dgms[1]

        if len(h1) == 0:
            return np.zeros(5)

        persistence = h1[:, 1] - h1[:, 0]

        return np.array([
            len(persistence),
            persistence.max(),
            persistence.mean(),
            persistence.sum(),
            persistence.std()
        ])

    except:
        return np.zeros(5)
print("Computing TDA...")

tda_full = np.zeros((X.shape[0], 5))

sample_nodes = list(node_map.keys())[:5000]  # txIds

for tx in sample_nodes:
    try:
        sub_nodes = list(nx.single_source_shortest_path_length(G, tx, cutoff=2).keys())
        subgraph = G.subgraph(sub_nodes)

        tda_feat = compute_tda(subgraph)

        idx = node_map[tx]  # map txId → index
        tda_full[idx] = tda_feat

    except:
        continue

# combine features
X = np.concatenate([X, tda_full], axis=1)

print("TDA done!")

Computing TDA...


/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


TDA done!


In [ ]:
# ================================
# CLEAN DATA BEFORE GNN
# ================================

# Replace inf → nan
X[np.isinf(X)] = np.nan

# Clip extreme values (VERY IMPORTANT)
X = np.clip(X, -1e6, 1e6)

# Replace nan → 0
X = np.nan_to_num(X, nan=0.0)

print("NaN count:", np.isnan(X).sum())
print("Inf count:", np.isinf(X).sum())

NaN count: 0
Inf count: 0


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.float)
y_tensor = torch.tensor(y, dtype=torch.float)

In [ ]:
# ================================
# 6. TRAIN TEST SPLIT
# ================================
train_idx, test_idx = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42)

X_tensor = torch.tensor(X, dtype=torch.float)
y_tensor = torch.tensor(y, dtype=torch.float)


In [ ]:
# ================================
# 7. GAT MODEL
# ================================
class GATNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.gat1 = GATConv(in_dim, 64, heads=4)
        self.gat2 = GATConv(256, 64, heads=4)
        self.lin = nn.Linear(256, 1)

    def forward(self, x, edge_index):
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        return self.lin(x).squeeze()   # ✅ NO sigmoid

model = GATNet(X.shape[1])


In [ ]:
# ================================
# 8. TRAIN GAT (FINAL STABLE)
# ================================

# Class imbalance handling
pos_weight = (len(y_tensor) - y_tensor.sum()) / y_tensor.sum()
pos_weight = torch.tensor(pos_weight, dtype=torch.float)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(300):
    model.train()
    optimizer.zero_grad()

    logits = model(X_tensor, edge_index)

    loss = criterion(logits[train_idx], y_tensor[train_idx])

    if torch.isnan(loss):
        print("NaN detected! Stopping training.")
        break

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

/tmp/ipykernel_619/2459378258.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight = torch.tensor(pos_weight, dtype=torch.float)


Epoch 0, Loss: 1.2674
Epoch 5, Loss: 0.8224
Epoch 10, Loss: 0.7088
Epoch 15, Loss: 0.6416
Epoch 20, Loss: 0.5851
Epoch 25, Loss: 0.5406
Epoch 30, Loss: 0.5025
Epoch 35, Loss: 0.4684
Epoch 40, Loss: 0.4377
Epoch 45, Loss: 0.4124
Epoch 50, Loss: 0.3893
Epoch 55, Loss: 0.3678
Epoch 60, Loss: 0.3473
Epoch 65, Loss: 0.3259
Epoch 70, Loss: 0.3050
Epoch 75, Loss: 0.2876
Epoch 80, Loss: 0.2716
Epoch 85, Loss: 0.2566
Epoch 90, Loss: 0.2429
Epoch 95, Loss: 0.2300
Epoch 100, Loss: 0.2178
Epoch 105, Loss: 0.2063
Epoch 110, Loss: 0.1954
Epoch 115, Loss: 0.1850
Epoch 120, Loss: 0.1752
Epoch 125, Loss: 0.1660
Epoch 130, Loss: 0.1580
Epoch 135, Loss: 0.1541
Epoch 140, Loss: 0.1478
Epoch 145, Loss: 0.1402
Epoch 150, Loss: 0.1343
Epoch 155, Loss: 0.1287
Epoch 160, Loss: 0.1239
Epoch 165, Loss: 0.1195
Epoch 170, Loss: 0.1151
Epoch 175, Loss: 0.1111
Epoch 180, Loss: 0.1071
Epoch 185, Loss: 0.1033
Epoch 190, Loss: 0.0997
Epoch 195, Loss: 0.0962
Epoch 200, Loss: 0.0928
Epoch 205, Loss: 0.0972
Epoch 210, Los

In [ ]:
# ================================
# 9. EVALUATE GAT (FIXED)
# ================================
model.eval()

with torch.no_grad():
    logits = model(X_tensor, edge_index)
    probs = torch.sigmoid(logits)
    pred = (probs > 0.5).int().numpy()

print("\nGAT Results:")
print("Accuracy:", accuracy_score(y[test_idx], pred[test_idx]))
print("F1:", f1_score(y[test_idx], pred[test_idx]))


GAT Results:
Accuracy: 0.9682164715988403
F1: 0.8458333333333333


In [ ]:
# ================================
# 9. EVALUATE GAT (trying using another way)
# ================================
model.eval()

with torch.no_grad():
    logits = model(X_tensor, edge_index)
    probs = torch.sigmoid(logits)
    pred = (probs > 0.3).int().numpy()

print("\nGAT Results:")
print("Accuracy:", accuracy_score(y[test_idx], pred[test_idx]))
print("F1:", f1_score(y[test_idx], pred[test_idx]))


GAT Results:
Accuracy: 0.959411575217438
F1: 0.8141592920353983


## AUC

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y[test_idx], probs[test_idx])
print("AUC:", auc)

AUC: 0.9811576552982701


In [ ]:
probs_np = probs.detach().cpu().numpy()

In [ ]:

from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1 = 0
best_thresh = 0

for t in thresholds:
  pred_temp = (probs > t).int()
  f1 = f1_score(y[test_idx], pred_temp[test_idx])

  if f1 > best_f1:
      best_f1 = f1
      best_thresh = t

print("Best Threshold:", best_thresh)
print("Best F1:", best_f1)

Best Threshold: 0.8500000000000002
Best F1: 0.8821855435401252


In [ ]:
# ================================
# 10. BASELINES (FINAL)
# ================================

from sklearn.impute import SimpleImputer

print("\nCleaning data for ML models...")

# Step 1: Replace inf with NaN
X[np.isinf(X)] = np.nan

# Step 2: Clip extreme values (safety)
X = np.clip(X, -1e6, 1e6)

# Step 3: Impute missing values
imputer = SimpleImputer(strategy="mean")
X_np = imputer.fit_transform(X)

# Final sanity check
print("NaN count:", np.isnan(X_np).sum())
print("Inf count:", np.isinf(X_np).sum())

# ----------------
# SVM
# ----------------
print("\nTraining SVM...")

svm = SVC()
svm.fit(X_np[train_idx], y[train_idx])
pred_svm = svm.predict(X_np[test_idx])

print("\nSVM Results:")
print("Accuracy:", accuracy_score(y[test_idx], pred_svm))
print("F1 Score:", f1_score(y[test_idx], pred_svm))

# ----------------
# Random Forest
# ----------------
print("\nTraining Random Forest...")

rf = RandomForestClassifier(n_estimators=50)
rf.fit(X_np[train_idx], y[train_idx])
pred_rf = rf.predict(X_np[test_idx])

print("\nRandom Forest Results:")
print("Accuracy:", accuracy_score(y[test_idx], pred_rf))
print("F1 Score:", f1_score(y[test_idx], pred_rf))


Cleaning data for ML models...
NaN count: 0
Inf count: 0

Training SVM...

SVM Results:
Accuracy: 0.9722967894341243
F1 Score: 0.8455089820359282

Training Random Forest...

Random Forest Results:
Accuracy: 0.9884033072049823
F1 Score: 0.9362455726092089


In [ ]:
# ================================
# 11. BiLSTM / BiGRU (FINAL)
# ================================

print("\nTraining Deep Learning models...")

# Convert cleaned data to tensor
X_tensor_clean = torch.tensor(X_np, dtype=torch.float)
y_tensor_clean = torch.tensor(y, dtype=torch.float)

# ----------------
# BiLSTM
# ----------------
class BiLSTM(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 64, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        out, _ = self.lstm(x)
        return torch.sigmoid(self.fc(out[:, -1])).squeeze()

# ----------------
# BiGRU
# ----------------
class BiGRU(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, 64, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        out, _ = self.gru(x)
        return torch.sigmoid(self.fc(out[:, -1])).squeeze()

# ----------------
# Training Function
# ----------------
def train_dl(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    for epoch in range(10):
        model.train()
        optimizer.zero_grad()

        out = model(X_tensor_clean[train_idx])
        loss = F.binary_cross_entropy(out, y_tensor_clean[train_idx])

        loss.backward()
        optimizer.step()

    return model

# ----------------
# Train LSTM
# ----------------
lstm = train_dl(BiLSTM(X_np.shape[1]))

pred_lstm = (lstm(X_tensor_clean).detach().numpy() > 0.5).astype(int)

print("\nBiLSTM Results:")
print("Accuracy:", accuracy_score(y[test_idx], pred_lstm[test_idx]))
print("F1 Score:", f1_score(y[test_idx], pred_lstm[test_idx]))

# ----------------
# Train GRU
# ----------------
gru = train_dl(BiGRU(X_np.shape[1]))

pred_gru = (gru(X_tensor_clean).detach().numpy() > 0.5).astype(int)

print("\nBiGRU Results:")
print("Accuracy:", accuracy_score(y[test_idx], pred_gru[test_idx]))
print("F1 Score:", f1_score(y[test_idx], pred_gru[test_idx]))


Training Deep Learning models...

BiLSTM Results:
Accuracy: 0.90475679158166
F1 Score: 0.051336898395721926

BiGRU Results:
Accuracy: 0.9180715129389027
F1 Score: 0.3484201537147737


## GAT without TDA

In [ ]:
X_original = X.copy()

In [ ]:
# ================================
# CLEAN DATA (NO TDA)
# ================================

X_no_tda = X_original.copy()

# Replace inf → nan
X_no_tda[np.isinf(X_no_tda)] = np.nan

# Clip extreme values
X_no_tda = np.clip(X_no_tda, -1e6, 1e6)

# Replace nan → 0
X_no_tda = np.nan_to_num(X_no_tda, nan=0.0)

# Normalize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_no_tda = scaler.fit_transform(X_no_tda)

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    np.arange(len(X_no_tda)),
    test_size=0.2,
    random_state=42
)

In [ ]:
X_tensor_no_tda = torch.tensor(X_no_tda, dtype=torch.float)
y_tensor = torch.tensor(y, dtype=torch.float)

In [ ]:
class GATNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.gat1 = GATConv(in_dim, 64, heads=4)
        self.gat2 = GATConv(256, 64, heads=4)
        self.lin = nn.Linear(256, 1)

    def forward(self, x, edge_index):
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        return self.lin(x).squeeze()   # NO sigmoid

In [ ]:
model = GATNet(X_no_tda.shape[1])

# Class imbalance
pos_weight = (len(y_tensor) - y_tensor.sum()) / y_tensor.sum()
pos_weight = torch.tensor(pos_weight, dtype=torch.float)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    model.train()
    optimizer.zero_grad()

    logits = model(X_tensor_no_tda, edge_index)

    loss = criterion(logits[train_idx], y_tensor[train_idx])

    if torch.isnan(loss):
        print("NaN detected! Stopping training.")
        break

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

/tmp/ipykernel_619/97104888.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight = torch.tensor(pos_weight, dtype=torch.float)


Epoch 0, Loss: 1.3255
Epoch 10, Loss: 0.7011
Epoch 20, Loss: 0.5771
Epoch 30, Loss: 0.4977
Epoch 40, Loss: 0.4374
Epoch 50, Loss: 0.3895
Epoch 60, Loss: 0.3488
Epoch 70, Loss: 0.3105
Epoch 80, Loss: 0.2767
Epoch 90, Loss: 0.2473


In [ ]:
model.eval()

with torch.no_grad():
    logits = model(X_tensor_no_tda, edge_index)
    probs = torch.sigmoid(logits)

probs_np = probs.detach().cpu().numpy()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Default threshold
pred = (probs_np > 0.5).astype(int)

print("\nGAT (NO TDA) Results:")
print("Accuracy:", accuracy_score(y[test_idx], pred[test_idx]))
print("F1:", f1_score(y[test_idx], pred[test_idx]))

auc = roc_auc_score(y[test_idx], probs_np[test_idx])
print("AUC:", auc)


GAT (NO TDA) Results:
Accuracy: 0.9393321163964351
F1: 0.7458389563652722
AUC: 0.9789303171550778


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Default threshold
pred = (probs_np > 0.5).astype(int)

print("\nGAT (NO TDA) Results:")
print("Accuracy:", accuracy_score(y[test_idx], pred[test_idx]))
print("F1:", f1_score(y[test_idx], pred[test_idx]))

auc = roc_auc_score(y[test_idx], probs_np[test_idx])
print("AUC:", auc)


GAT (NO TDA) Results:
Accuracy: 0.9393321163964351
F1: 0.7458389563652722
AUC: 0.9789303171550778
